# ChestMedicalNet Training (Colab)

Trains ChestMedicalNet: a ConvNeXt-Base backbone + Feature Pyramid Network + CBAM attention + disease-specific pathways (TB / Pneumonia / General) + Bayesian uncertainty + CORAL ordinal severity head, on NIH ChestX-ray14.

**Read this before running:**
- NIH ChestX-ray14 has no TB label. The TB pathway exists architecturally but trains as a no-op (masked loss) on this dataset -- see `config/config.py::TB_LABEL_AVAILABLE`. Do not treat its output as a real TB classifier.
- ConvNeXt-Base is ~91M params total with the FPN/attention/heads added. This fits a free-tier T4 (16GB) with a modest batch size, but watch for OOM -- reduce `--batch-size` if you hit one.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine, A100/L4 is faster). Run the next cell first and confirm `CUDA available: True` before doing anything else -- there's no point downloading 45GB onto a CPU-only runtime.

**One-time setup you need outside this notebook:**
1. A free Kaggle account + API token: kaggle.com -> Settings -> Create New Token (downloads `kaggle.json`).
2. In this Colab notebook, click the key icon (Secrets) in the left sidebar and add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from `kaggle.json`). This avoids re-uploading credentials every session.
3. Replace `REPO_URL` in the cell below with your GitHub repo URL.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('torch', torch.__version__, '| torchvision', end=' ')
import torchvision; print(torchvision.__version__)

## 1. Get the code

In [ ]:
REPO_URL = "REPLACE_WITH_YOUR_GITHUB_REPO_URL"  # e.g. https://github.com/<you>/ChestXRayMaxVit.git
REPO_DIR = "/content/ChestXRayMaxVit"

!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -q -r requirements-colab.txt

## 2. Mount Google Drive (for checkpoints that survive a disconnected session)

Checkpoints go to Drive, not the Colab VM's local disk -- the VM (and everything on it) is wiped when the session ends, but the dataset is re-downloaded fresh each session anyway (see step 3), so only checkpoints/logs need to persist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/ChestXRayMaxViTv2/checkpoints'
LOG_DIR = '/content/drive/MyDrive/ChestXRayMaxViTv2/logs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.environ['CHECKPOINT_DIR'] = CHECKPOINT_DIR
os.environ['LOG_DIR'] = LOG_DIR

## 3. Download the dataset from Kaggle

Uses `kagglehub` (Colab's current recommended Kaggle integration) rather than the older `kaggle` CLI. `kagglehub.dataset_download` caches under `/root/.cache/kagglehub/...` and the exact path can shift between dataset versions, so instead of hardcoding it, the cell below searches whatever path it returns for `Data_Entry_2017.csv` and resolves `ARCHIVE_DIR` from that -- every later cell reads `$ARCHIVE_DIR` rather than a hardcoded path.

This is the same `nih-chest-xrays/data` dataset as the local `archive/` folder (same file names: `Data_Entry_2017.csv`, `images_001/images/` ... `images_012/images/`), so no code changes are needed.

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import glob
import kagglehub

raw_path = kagglehub.dataset_download("nih-chest-xrays/data")
print("kagglehub download path:", raw_path)

matches = glob.glob(os.path.join(raw_path, "**", "Data_Entry_2017.csv"), recursive=True)
assert matches, f"Data_Entry_2017.csv not found anywhere under {raw_path} -- download may have failed, check output above."
ARCHIVE_DIR = os.path.dirname(matches[0])
os.environ['ARCHIVE_DIR'] = ARCHIVE_DIR
print("Resolved ARCHIVE_DIR:", ARCHIVE_DIR)

In [ ]:
# Sanity-check the download landed in the expected layout.
!ls "$ARCHIVE_DIR" | head -20
!test -f "$ARCHIVE_DIR/Data_Entry_2017.csv" && echo 'Data_Entry_2017.csv found'
!ls "$ARCHIVE_DIR"/images_001/images | head -3

## 4. Pipeline sanity check (recommended before the full run)

A few epochs on a small subset -- confirms the full pipeline (data loading, model, loss, GPU training loop) works on this environment before committing GPU-hours to the full 100-epoch schedule.

In [ ]:
!python sanity_check.py --archive-dir "$ARCHIVE_DIR" --num-workers 2 --device cuda

## 5. Full training run

AdamW + cosine schedule with warmup, class-balanced focal loss, early stopping (patience 10) on validation mean per-class F1, up to 100 epochs. Checkpoints (`last_checkpoint.pt` every epoch, `best_model.pt` on improvement) go to Drive. Add `--batch-size N` to override the default if you hit an out-of-memory error.

In [ ]:
!python main.py train --model chest --architecture custom \
  --archive-dir "$ARCHIVE_DIR" \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --log-dir "$LOG_DIR" \
  --num-workers 2 \
  --device cuda

## 6. Resuming after a disconnected session

Colab free-tier sessions disconnect after ~12h (or sooner if idle). Re-run steps 1-3 (code + dataset are gone when the VM is recycled -- Drive is not, so `$CHECKPOINT_DIR` still has `last_checkpoint.pt`), then resume instead of restarting from epoch 0:

In [ ]:
!python main.py train --model chest --architecture custom \
  --archive-dir "$ARCHIVE_DIR" \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --log-dir "$LOG_DIR" \
  --num-workers 2 \
  --device cuda \
  --resume "$CHECKPOINT_DIR/last_checkpoint.pt"